# Faturamento — Análise de Pre-invoices
Análise semanal de pré-invoices de frete por status de pagamento e tipo de provedor.

**Como usar:** Defina o `TIPO` na célula abaixo e execute tudo (`Run All`).

In [ ]:
# ============================================================
# CONFIGURAÇÃO — altere aqui antes de rodar
# ============================================================
# Opções: 'regular' | 'complementar' | 'ambos'
TIPO = 'regular'
# ============================================================

In [ ]:
from google.cloud import bigquery
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='meli-bi-data')

# Monta o filtro de tipo
if TIPO == 'regular':
    filtro_tipo = "WHERE b.SHP_SRM_PRIVN_PRE_INVOICE_TYPE = 'regular'"
elif TIPO == 'complementar':
    filtro_tipo = "WHERE b.SHP_SRM_PRIVN_PRE_INVOICE_TYPE = 'complementar'"
else:
    filtro_tipo = ''  # ambos — sem filtro

query = f"""
WITH TopPeriods AS (
  SELECT DISTINCT JSON_EXTRACT_SCALAR(SHP_SRM_PRIVN_PERIOD, '$.name') AS Name
  FROM WHOWNER.BT_SRM_PRE_INVOICE_ASSIGNED_EVENT_COST_ENTITIES
  WHERE SHP_SRM_PRIVN_EVENT_STATUS = 'PROCESSED'
  ORDER BY Name DESC
  LIMIT 5
),

LatestVersion AS (
  SELECT
    SHP_SRM_PRIVN_TRANSACTION_ID,
    MAX(SHP_SRM_PRIVN_VERSION_ID) AS MaxVersion
  FROM WHOWNER.BT_SRM_PRE_INVOICE_ASSIGNED_EVENT_COST_ENTITIES
  WHERE SHP_SRM_PRIVN_EVENT_STATUS = 'PROCESSED'
  GROUP BY SHP_SRM_PRIVN_TRANSACTION_ID
),

Base AS (
  SELECT
    C.SHP_SRM_PRIVN_TRANSACTION_ID,
    JSON_EXTRACT_SCALAR(C.SHP_SRM_PRIVN_PERIOD, '$.name') AS Name,
    C.SHP_SRM_PRIVN_COST_AMT,
    C.SHP_SRM_PRIVN_PRE_INVOICE_TYPE,
    C.SHP_SRM_PRIVN_PRE_INVOICE_ID,
    C.SHP_SRM_PRIVN_PAYLOAD,
    PRV.TYPE AS PROVIDER_TYPE,
    CASE
      WHEN PRV.TYPE = 'TAC-CPF' AND T.INVOICE_IDENTIFIER_STATUS IS NULL THEN 'nao_emite_nfs'
      WHEN T.INVOICE_IDENTIFIER_STATUS IN ('sap_status_pago_realizado', 'sap_status_pagado') THEN 'sap_status_pago_realizado'
      ELSE T.INVOICE_IDENTIFIER_STATUS
    END AS INVOICE_IDENTIFIER_STATUS
  FROM WHOWNER.BT_SRM_PRE_INVOICE_ASSIGNED_EVENT_COST_ENTITIES C
  JOIN LatestVersion LV
    ON C.SHP_SRM_PRIVN_TRANSACTION_ID = LV.SHP_SRM_PRIVN_TRANSACTION_ID
   AND C.SHP_SRM_PRIVN_VERSION_ID = LV.MaxVersion
  JOIN TopPeriods TP
    ON JSON_EXTRACT_SCALAR(C.SHP_SRM_PRIVN_PERIOD, '$.name') = TP.Name
  JOIN WHOWNER.BT_SHP_SRM_PROVIDERS PRV
    ON C.SHP_SRM_PRIVN_PROVIDER_ID = PRV.ID
  LEFT JOIN WHOWNER.BT_SRM_INVOICE_FILE_TRACEABILITY T
    ON CAST(C.SHP_SRM_PRIVN_PRE_INVOICE_ID AS STRING) = T.PREINVOICE_ID
  WHERE C.SHP_SRM_PRIVN_EVENT_STATUS = 'PROCESSED'
    AND PRV.TYPE IN ('TAC-CNPJ', 'TAC-CPF', 'ETC', 'MEI')
),

PayloadKeyVals AS (
  SELECT
    b.SHP_SRM_PRIVN_TRANSACTION_ID,
    MAX(IF(JSON_EXTRACT_SCALAR(p, '$.key') = 'external_route_id', JSON_EXTRACT_SCALAR(p, '$.value'), NULL)) AS external_route_id
  FROM Base b,
       UNNEST(JSON_EXTRACT_ARRAY(b.SHP_SRM_PRIVN_PAYLOAD, '$.payload')) AS p
  GROUP BY b.SHP_SRM_PRIVN_TRANSACTION_ID
)

SELECT
  b.Name,
  b.SHP_SRM_PRIVN_PRE_INVOICE_TYPE,
  b.INVOICE_IDENTIFIER_STATUS,
  COUNT(DISTINCT p.external_route_id)            AS qtd_rotas_distintas,
  COUNT(DISTINCT b.SHP_SRM_PRIVN_PRE_INVOICE_ID) AS qtd_pre_invoice_id_distintos,
  SUM(b.SHP_SRM_PRIVN_COST_AMT)                  AS SHP_SRM_PRIVN_COST_AMT
FROM Base b
LEFT JOIN PayloadKeyVals p
  ON b.SHP_SRM_PRIVN_TRANSACTION_ID = p.SHP_SRM_PRIVN_TRANSACTION_ID
{filtro_tipo}
GROUP BY b.Name, b.SHP_SRM_PRIVN_PRE_INVOICE_TYPE, b.INVOICE_IDENTIFIER_STATUS
ORDER BY b.Name ASC, b.SHP_SRM_PRIVN_PRE_INVOICE_TYPE, b.INVOICE_IDENTIFIER_STATUS
"""

print(f'Executando query — tipo: {TIPO} ...')
df = client.query(query).to_dataframe()
print(f'Linhas retornadas: {len(df)}')

In [ ]:
query_rv = f"""
WITH TopPeriods AS (
  SELECT DISTINCT JSON_EXTRACT_SCALAR(SHP_SRM_PRIVN_PERIOD, '$.name') AS Name
  FROM WHOWNER.BT_SRM_PRE_INVOICE_ASSIGNED_EVENT_COST_ENTITIES
  WHERE SHP_SRM_PRIVN_EVENT_STATUS = 'PROCESSED'
  ORDER BY Name DESC
  LIMIT 5
),
LatestVersion AS (
  SELECT SHP_SRM_PRIVN_TRANSACTION_ID, MAX(SHP_SRM_PRIVN_VERSION_ID) AS MaxVersion
  FROM WHOWNER.BT_SRM_PRE_INVOICE_ASSIGNED_EVENT_COST_ENTITIES
  WHERE SHP_SRM_PRIVN_EVENT_STATUS = 'PROCESSED'
  GROUP BY SHP_SRM_PRIVN_TRANSACTION_ID
),
Base AS (
  SELECT C.SHP_SRM_PRIVN_TRANSACTION_ID,
         JSON_EXTRACT_SCALAR(C.SHP_SRM_PRIVN_PERIOD, '$.name') AS Name,
         C.SHP_SRM_PRIVN_PRE_INVOICE_TYPE,
         C.SHP_SRM_PRIVN_PAYLOAD
  FROM WHOWNER.BT_SRM_PRE_INVOICE_ASSIGNED_EVENT_COST_ENTITIES C
  JOIN LatestVersion LV ON C.SHP_SRM_PRIVN_TRANSACTION_ID = LV.SHP_SRM_PRIVN_TRANSACTION_ID
                        AND C.SHP_SRM_PRIVN_VERSION_ID = LV.MaxVersion
  JOIN TopPeriods TP ON JSON_EXTRACT_SCALAR(C.SHP_SRM_PRIVN_PERIOD, '$.name') = TP.Name
  JOIN WHOWNER.BT_SHP_SRM_PROVIDERS PRV ON C.SHP_SRM_PRIVN_PROVIDER_ID = PRV.ID
  WHERE C.SHP_SRM_PRIVN_EVENT_STATUS = 'PROCESSED'
    AND PRV.TYPE IN ('TAC-CNPJ', 'TAC-CPF', 'ETC', 'MEI')
),
PayloadRV AS (
  SELECT b.SHP_SRM_PRIVN_TRANSACTION_ID,
         b.Name,
         b.SHP_SRM_PRIVN_PRE_INVOICE_TYPE,
         MAX(IF(JSON_EXTRACT_SCALAR(p, '$.key') = 'external_route_id', JSON_EXTRACT_SCALAR(p, '$.value'), NULL)) AS external_route_id,
         MAX(IF(JSON_EXTRACT_SCALAR(p, '$.key') = 'vehicle_id',        JSON_EXTRACT_SCALAR(p, '$.value'), NULL)) AS vehicle_id
  FROM Base b, UNNEST(JSON_EXTRACT_ARRAY(b.SHP_SRM_PRIVN_PAYLOAD, '$.payload')) AS p
  GROUP BY b.SHP_SRM_PRIVN_TRANSACTION_ID, b.Name, b.SHP_SRM_PRIVN_PRE_INVOICE_TYPE
)
SELECT Name,
       COUNT(DISTINCT external_route_id) AS qtd_rotas_distintas,
       COUNT(DISTINCT vehicle_id)        AS qtd_veiculos_distintos
FROM PayloadRV
{filtro_tipo.replace('b.', '')}
GROUP BY Name
ORDER BY Name DESC
"""

print(f'Executando query rotas/veículos — tipo: {TIPO} ...')
df_rv = client.query(query_rv).to_dataframe()
print(f'Linhas retornadas: {len(df_rv)}')

In [ ]:
# ============================================================
# Mapeamento de status para as linhas dos quadros
# ============================================================

OUTROS = [
    'accounting_document_update', 'accounting_error', 'accounting_sent_sap',
    'escalated_to_external_team', 'validate_files_fail',
    'validate_provision_success', 'preconciliation_ok_invoice'
]

ORDEM_STATUS = [
    'sap_status_pago_realizado',
    'sap_status_liberado_para_el_pago',
    'sap_status_en_proceso_de_pago',
    'nao_emite_nfs',
    'preconciliation_fail_invoice',
    'NULL',
    'Outros ¹',
    'TOTAL',
]

def normalizar_status(s):
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return 'NULL'
    if s in OUTROS:
        return 'Outros ¹'
    return s

df['status_norm'] = df['INVOICE_IDENTIFIER_STATUS'].apply(normalizar_status)

# Períodos ordenados do mais antigo ao mais novo
periodos = sorted(df['Name'].unique())

def build_pivot(metric):
    agg = df.groupby(['Name', 'status_norm'])[metric].sum().reset_index()
    piv = agg.pivot(index='status_norm', columns='Name', values=metric).fillna(0)
    # Adiciona linhas faltantes
    for s in ORDEM_STATUS[:-1]:
        if s not in piv.index:
            piv.loc[s] = 0
    # Linha TOTAL
    piv.loc['TOTAL'] = piv.sum()
    piv = piv.loc[ORDEM_STATUS]
    piv = piv[periodos]  # colunas do mais antigo ao mais novo
    return piv

piv_qtd  = build_pivot('qtd_pre_invoice_id_distintos')
piv_custo = build_pivot('SHP_SRM_PRIVN_COST_AMT')

# % sobre total do período
piv_pct = piv_qtd.div(piv_qtd.loc['TOTAL']).mul(100).round(1)
piv_pct.loc['TOTAL'] = 100.0

print('Dados processados.')

In [ ]:
# ============================================================
# Quadros 4 e 5 — Rotas e Veículos distintos por período
# ============================================================

# Ordena períodos mais antigo → mais novo
df_rv_sorted = df_rv.sort_values('Name')
periodos_rv = list(df_rv_sorted['Name'])

def render_table_simple(data_dict, title):
    """Renderiza tabela com métricas como linhas e períodos como colunas."""
    header = '<tr><th>Métrica</th>' + ''.join(f'<th>{p}</th>' for p in periodos_rv) + '</tr>'
    rows_html = ''
    vals_indexed = df_rv_sorted.set_index('Name')
    for label, col in data_dict.items():
        cells = ''.join(
            f'<td align="right">{int(vals_indexed.loc[p, col]):,}'.replace(',', '.') + '</td>'
            for p in periodos_rv
        )
        rows_html += f'<tr><td>{label}</td>{cells}</tr>'
    display(HTML(f'<h3>{title}</h3><table border="1" cellpadding="5" cellspacing="0">{header}{rows_html}</table>'))

render_table_simple(
    {'Rotas distintas': 'qtd_rotas_distintas'},
    f'Quadro 4 — Rotas distintas por período ({TIPO})'
)

render_table_simple(
    {'Veículos distintos': 'qtd_veiculos_distintos'},
    f'Quadro 5 — Veículos distintos por período ({TIPO})'
)

In [ ]:
# ============================================================
# Quadro 1 — Quantidade de Pre-invoices
# ============================================================
from IPython.display import display, HTML

def fmt_qtd(v):
    return '-' if v == 0 else f'{int(v):,}'.replace(',', '.')

def fmt_pct(v, idx):
    if idx == 'TOTAL':
        return '100%'
    return '-' if v == 0 else f'{v:.1f}%'

def fmt_custo(v):
    return '-' if v == 0 else f'R$ {v:,.0f}'.replace(',', '.')

def render_table(piv, fmt_fn, title, nota=None):
    cols = list(piv.columns)
    header = '<tr><th>' + '</th><th>'.join(['Status'] + cols) + '</th></tr>'
    rows_html = ''
    for idx, row in piv.iterrows():
        bold = ' style="font-weight:bold"' if idx == 'TOTAL' else ''
        cells = ''.join(f'<td align="right">{fmt_fn(row[c], idx) if callable(fmt_fn) and fmt_fn.__code__.co_varnames[:2] == ("v", "idx") else fmt_fn(row[c])}</td>' for c in cols)
        rows_html += f'<tr{bold}><td>{idx}</td>{cells}</tr>'
    html = f'<h3>{title}</h3><table border="1" cellpadding="5" cellspacing="0">{header}{rows_html}</table>'
    if nota:
        html += f'<p style="font-size:0.85em">{nota}</p>'
    display(HTML(html))

nota_outros = '¹ Outros: accounting_document_update, accounting_error, accounting_sent_sap, escalated_to_external_team, validate_files_fail, validate_provision_success, preconciliation_ok_invoice'

render_table(
    piv_qtd,
    fmt_qtd,
    f'Quadro 1 — Quantidade de Pre-invoices ({TIPO})'
)

render_table(
    piv_pct,
    lambda v, idx: ('100%' if idx == 'TOTAL' else ('-' if v == 0 else f'{v:.1f}%')),
    f'Quadro 2 — % sobre total de pre-invoices do período ({TIPO})'
)

render_table(
    piv_custo,
    fmt_custo,
    f'Quadro 3 — Custo total em R$ ({TIPO})',
    nota=nota_outros
)